# Validação externa (Messidor-2)

Mede o modelo escolhido (R12 semente 42) uma única vez no Messidor-2: 1.748 fotos de 874 exames com os dois
olhos, de outra população e de outra câmera, que nunca entraram no treino. Nada é escolhido aqui: os limiares
são os que a validação do EyePACS escolheu no R09 (conferidos com o resumo dele), e só as fotos graduáveis
são medidas.

**Desfecho principal (fixado antes de rodar):** por paciente, só os pacientes com os dois olhos graduáveis,
qualquer retinopatia, sensibilidade e especificidade nos limiares do EyePACS e ROC-AUC, com intervalos de 95%
por bootstrap. O resto (por olho, por grau, referenciável, modos de TTA) é secundário.

Precisa, na pasta Datasets do Drive: `Messidor-2/` (as 4 partes `IMAGES.zip.00x` e o `messidor-2.csv`) e
`Messidor2_rotulos/` (graus do Google, baixados pelo Baixar_Datasets). A lógica está em
`src/retinopatia/validacao_externa.py`. Sessão A100 curta; a sessão desliga sozinha no fim.

## Preparação do repositório

In [ ]:
# O código do projeto vive no repositório do GitHub.
# Esta célula traz (ou atualiza) esse código dentro do Colab.

import os
import sys

URL_REPOSITORIO = "https://github.com/Rcorrea07/ML_Retinopatia.git"
PASTA_REPOSITORIO = "/content/ML_Retinopatia"

# Versão do código: "origin/main" é a mais recente; uma branch
# ("origin/refatoracao"), uma tag ou um commit reproduz outra. O commit de
# cada treino fica no configuracao.json da pasta de resultados.
REVISAO_CODIGO = "origin/main"  #@param {type:"string"}

# Validação externa do modelo escolhido (CAMINHO_MODELO_ESCOLHIDO em
# configuracao.py). Sem treino e sem o teste do EyePACS.
EXPERIMENTO = "V01_r12_messidor2"  #@param {type:"string"}
os.environ["RETINOPATIA_EXPERIMENTO"] = EXPERIMENTO

os.environ["RETINOPATIA_FUMACA"] = "0"

# Rode UMA VEZ SÓ: medir de novo depois de mexer em algo transforma a
# validação externa em mais uma validação.
os.environ["RETINOPATIA_AVALIAR_TESTE"] = "0"

# True = encerra a sessão no fim, também se der erro
DESLIGAR_AO_TERMINAR = True  #@param {type:"boolean"}

if not os.path.exists(PASTA_REPOSITORIO):
    !git clone {URL_REPOSITORIO} {PASTA_REPOSITORIO}

!git -C {PASTA_REPOSITORIO} fetch --tags origin
!git -C {PASTA_REPOSITORIO} checkout --detach {REVISAO_CODIGO}
!git -C {PASTA_REPOSITORIO} log -1 --date=short --format="Código no commit %h (%cd): %s"

# Torna o pacote "retinopatia" importável neste notebook
caminho_src = os.path.join(PASTA_REPOSITORIO, "src")

if caminho_src not in sys.path:
    sys.path.insert(0, caminho_src)

# Recarrega os módulos automaticamente quando esta célula traz código
# novo, evitando ter que reiniciar o ambiente a cada edição no VS Code.
# O Colab usa Python 3.12+, que não tem mais o módulo "imp", mas a
# versão do IPython instalada nele ainda o importa no autoreload.
# Um substituto mínimo resolve; se mesmo assim falhar, o notebook
# segue sem autoreload (aí é preciso reiniciar após cada git pull).
import importlib
import types

try:
    import imp
except ModuleNotFoundError:
    sys.modules["imp"] = types.ModuleType("imp")
    sys.modules["imp"].reload = importlib.reload

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception as erro:
    print("Autoreload indisponível:", erro)

print("Repositório pronto em:", PASTA_REPOSITORIO)

## Conexão com o Drive

In [ ]:
from retinopatia import ambiente_colab, validacao_externa

ambiente_colab.montar_drive()

## Messidor-2

In [ ]:
resultado = validacao_externa.rodar_messidor2(
    desligar_ao_terminar=DESLIGAR_AO_TERMINAR
)